# Class Attributes — Advanced Tutorial Problems with Solutions

This notebook is a **second, independent advanced practice notebook** on Python class attributes.

The style is deliberately tutorial-oriented:

- introduce one idea at a time,
- run a small experiment,
- inspect what Python actually stored,
- explain the result,
- then solve a more challenging problem built from those observations.

The main theme throughout the notebook is:

> **Attribute access and namespace storage are related, but they are not the same thing.**

We will repeatedly ask two separate questions:

1. **What value does Python resolve when we write `obj.attr` or `Class.attr`?**
2. **Where is that name actually stored?**

That distinction is the key to understanding advanced class-attribute behavior.

## What we will practice

We will work with:

- class attributes created in a class body,
- attributes created dynamically at runtime,
- `getattr`, `setattr`, `delattr`, and dotted notation,
- `Class.__dict__`,
- `mappingproxy`,
- direct namespace membership,
- instance shadowing,
- inherited class attributes,
- subclass overrides,
- mutable class attributes,
- class methods,
- method objects stored in a class namespace,
- the method resolution order (MRO),
- reversible class configuration,
- and defensive class-level design patterns.

Every major section contains an advanced problem and a complete solution.

# Problem 1 — Reconstructing a Class Namespace

Suppose we define a class with several class attributes.

Before doing anything advanced, we need a reliable mental model of what the class body creates.

In [1]:
class Runtime:
    language = "Python"
    version = "3.12"
    implementation = "CPython"

We can access these values using dotted notation:

In [2]:
Runtime.language, Runtime.version, Runtime.implementation

('Python', '3.12', 'CPython')

Now inspect the class namespace itself.

In [3]:
Runtime.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'language': 'Python',
              'version': '3.12',
              'implementation': 'CPython',
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Runtime' objects>,
              '__weakref__': <attribute '__weakref__' of 'Runtime' objects>,
              '__doc__': None})

The result is a `mappingproxy`.

It contains the names created directly by the class body, together with several entries Python added automatically.

Let's isolate only the three names we created.

In [4]:
{
    name: Runtime.__dict__[name]
    for name in ("language", "version", "implementation")
}

{'language': 'Python', 'version': '3.12', 'implementation': 'CPython'}

### Step 1 — Direct membership

A useful question is not merely whether an attribute can be accessed, but whether it is stored **directly** in this class namespace.

In [5]:
"language" in Runtime.__dict__

True

Now compare that with `__name__`.

In [6]:
Runtime.__name__, "__name__" in Runtime.__dict__

('Runtime', False)

This is an important result.

`Runtime.__name__` exists, but `"__name__" in Runtime.__dict__` is false.

So direct namespace membership and attribute accessibility are not identical tests.

### Challenge

Write a function `describe_direct_attribute(cls, name)` that reports:

- whether the name is directly stored in `cls.__dict__`,
- the direct value if present,
- and the normally resolved value if accessible.

In [7]:
# Solution

_MISSING = object()

def describe_direct_attribute(cls, name):
    direct = name in cls.__dict__
    direct_value = cls.__dict__.get(name, _MISSING)
    resolved_value = getattr(cls, name, _MISSING)

    return {
        "name": name,
        "stored_directly": direct,
        "direct_value": (
            "<missing>"
            if direct_value is _MISSING
            else direct_value
        ),
        "resolved_value": (
            "<missing>"
            if resolved_value is _MISSING
            else resolved_value
        ),
    }

describe_direct_attribute(Runtime, "language")

{'name': 'language',
 'stored_directly': True,
 'direct_value': 'Python',
 'resolved_value': 'Python'}

In [8]:
describe_direct_attribute(Runtime, "__name__")

{'name': '__name__',
 'stored_directly': False,
 'direct_value': '<missing>',
 'resolved_value': 'Runtime'}

In [9]:
describe_direct_attribute(Runtime, "does_not_exist")

{'name': 'does_not_exist',
 'stored_directly': False,
 'direct_value': '<missing>',
 'resolved_value': '<missing>'}

### Takeaway

When debugging class attributes, distinguish:

- **namespace inspection**: `name in cls.__dict__`
- **attribute resolution**: `getattr(cls, name)`

They answer different questions.

# Problem 2 — Building a Dynamic Configuration Class

Python lets us add class attributes after the class has already been created.

This is useful in configuration systems, plugin systems, test fixtures, and metaprogramming.

In [10]:
class AppConfig:
    environment = "development"

We can add a new attribute with dotted assignment:

In [11]:
AppConfig.debug = True

In [12]:
AppConfig.debug

True

And the new name appears in the class namespace:

In [13]:
"debug" in AppConfig.__dict__

True

When the attribute name is stored in a string, dotted notation cannot be written directly.

That is where `setattr` becomes useful.

In [14]:
attribute_name = "workers"
setattr(AppConfig, attribute_name, 4)

In [15]:
AppConfig.workers

4

### Step 1 — Apply many values

Suppose configuration arrives as a dictionary:

In [16]:
incoming_config = {
    "timeout": 15,
    "retry_limit": 5,
    "region": "eu",
}

We can apply it one attribute at a time.

In [17]:
for name, value in incoming_config.items():
    setattr(AppConfig, name, value)

In [18]:
{
    name: getattr(AppConfig, name)
    for name in incoming_config
}

{'timeout': 15, 'retry_limit': 5, 'region': 'eu'}

### Step 2 — Make the operation safer

Blindly assigning arbitrary names can overwrite existing class behavior.

For example, a dynamic configuration loader should usually reject names beginning with `_`.

It may also be useful to reject names that already exist directly in the class namespace.

### Challenge

Create `safe_apply_config(cls, config)`.

Rules:

1. every name must be a valid Python identifier,
2. private or dunder-style names are rejected,
3. existing direct attributes are not overwritten,
4. accepted values are added with `setattr`,
5. return the names that were added.

In [19]:
# Solution

def safe_apply_config(cls, config):
    added = []

    for name, value in config.items():
        if not isinstance(name, str) or not name.isidentifier():
            raise ValueError(f"Invalid attribute name: {name!r}")

        if name.startswith("_"):
            raise ValueError(f"Private name is not allowed: {name!r}")

        if name in cls.__dict__:
            raise ValueError(
                f"{name!r} already exists directly on {cls.__name__}"
            )

        setattr(cls, name, value)
        added.append(name)

    return added

In [20]:
class SafeConfig:
    mode = "normal"

safe_apply_config(
    SafeConfig,
    {
        "timeout": 30,
        "workers": 8,
    },
)

['timeout', 'workers']

In [21]:
SafeConfig.timeout, SafeConfig.workers

(30, 8)

Let's verify that unsafe updates are rejected.

In [22]:
try:
    safe_apply_config(SafeConfig, {"mode": "dangerous"})
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

ValueError: 'mode' already exists directly on SafeConfig


In [23]:
try:
    safe_apply_config(SafeConfig, {"__dict__": 123})
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)

ValueError: Private name is not allowed: '__dict__'


### Takeaway

Use:

- dotted assignment for fixed attribute names,
- `setattr` when the name is dynamic,
- validation when names originate outside trusted source code.

# Problem 3 — Understanding `mappingproxy`

A class namespace is exposed through `Class.__dict__`.

But Python does not give us an ordinary mutable dictionary.

In [24]:
class State:
    status = "starting"

namespace = State.__dict__

type(namespace)

mappingproxy

Let's try reading through the mapping.

In [25]:
namespace["status"]

'starting'

Reading works normally.

Now try modifying it directly.

In [26]:
try:
    namespace["status"] = "ready"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: 'mappingproxy' object does not support item assignment


The class namespace view is read-only.

But this does **not** mean the class is immutable.

We can still modify the class using supported attribute operations.

In [27]:
State.status = "ready"
State.status

'ready'

Now inspect the same `namespace` object we saved earlier.

In [28]:
namespace["status"]

'ready'

The old `mappingproxy` reference sees the new value.

So the proxy is not a frozen copy. It is a read-only **view** of the live class namespace.

### Challenge

Prove the same thing for a newly created attribute.

Keep one reference to the mapping proxy, add a new class attribute with `setattr`, and verify that the proxy immediately exposes it.

In [29]:
# Solution

class LiveView:
    x = 10

proxy = LiveView.__dict__

print("before:", "y" in proxy)

setattr(LiveView, "y", 20)

print("after:", "y" in proxy)
print("value:", proxy["y"])

assert proxy["y"] == 20

before: False
after: True
value: 20


### Takeaway

`mappingproxy` protects the namespace from direct dictionary mutation while still reflecting legitimate changes to the underlying class.

# Problem 4 — Deleting Attributes and Observing Namespace Changes

We can create and mutate class attributes at runtime.

We can also delete them.

In [30]:
class Feature:
    enabled = True
    timeout = 10

Delete one attribute with `del`.

In [31]:
del Feature.timeout

In [32]:
"timeout" in Feature.__dict__

False

Delete another attribute with `delattr`.

In [33]:
delattr(Feature, "enabled")

In [34]:
Feature.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'Feature' objects>,
              '__weakref__': <attribute '__weakref__' of 'Feature' objects>,
              '__doc__': None})

### Step 1 — Safe deletion

Calling `delattr` for a missing name raises `AttributeError`.

In [35]:
try:
    delattr(Feature, "missing")
except AttributeError as exc:
    print(type(exc).__name__ + ":", exc)

AttributeError: type object 'Feature' has no attribute 'missing'


### Challenge

Write `delete_direct_attribute(cls, name)` that deletes a name **only if it is stored directly on the class**.

Return `True` when deletion occurred and `False` otherwise.

In [36]:
# Solution

def delete_direct_attribute(cls, name):
    if name not in cls.__dict__:
        return False

    delattr(cls, name)
    return True

In [37]:
class DeleteDemo:
    a = 1
    b = 2

delete_direct_attribute(DeleteDemo, "a")

True

In [38]:
delete_direct_attribute(DeleteDemo, "a")

False

In [39]:
DeleteDemo.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'b': 2,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'DeleteDemo' objects>,
              '__weakref__': <attribute '__weakref__' of 'DeleteDemo' objects>,
              '__doc__': None})

Why inspect `cls.__dict__` instead of merely using `hasattr`?

We will see the answer when inheritance enters the picture.

# Problem 5 — Class Attributes Seen Through Instances

Consider a class attribute:

In [40]:
class Tax:
    rate = 0.20

Create two instances.

In [41]:
a = Tax()
b = Tax()

Neither instance has an `rate` entry in its own namespace:

In [42]:
a.__dict__, b.__dict__

({}, {})

Yet both instances can access `rate`.

In [43]:
a.rate, b.rate

(0.2, 0.2)

The value is being found through the class.

Now change the class attribute.

In [44]:
Tax.rate = 0.25

In [45]:
a.rate, b.rate

(0.25, 0.25)

Both instances now resolve the updated class value.

### Step 1 — Create an instance shadow

Assign `rate` only to `a`.

In [46]:
a.rate = 0.50

In [47]:
Tax.rate, a.rate, b.rate

(0.25, 0.5, 0.25)

Now inspect `a.__dict__`.

In [48]:
a.__dict__

{'rate': 0.5}

`a` contains its own `rate`, so instance lookup finds that value first.

`b` has no local `rate`, so it still resolves the class value.

### Challenge

Write a function that reports whether a resolved attribute is currently shadowed by the instance.

In [49]:
# Solution

def instance_shadow_report(obj, name):
    instance_dict = getattr(obj, "__dict__", {})
    cls = type(obj)

    return {
        "resolved_value": getattr(obj, name),
        "stored_on_instance": name in instance_dict,
        "stored_directly_on_class": name in cls.__dict__,
    }

instance_shadow_report(a, "rate")

{'resolved_value': 0.5,
 'stored_on_instance': True,
 'stored_directly_on_class': True}

In [50]:
instance_shadow_report(b, "rate")

{'resolved_value': 0.25,
 'stored_on_instance': False,
 'stored_directly_on_class': True}

### Step 2 — Reveal the class attribute again

Delete the instance attribute.

In [51]:
del a.rate

In [52]:
a.rate, a.__dict__

(0.25, {})

After the shadow is removed, normal class lookup becomes visible again.

# Problem 6 — The `+=` Shadowing Trap

This pattern looks harmless:

In [53]:
class Counter:
    value = 0

c = Counter()

Now execute:

In [54]:
c.value += 1

What changed?

Let's inspect both namespaces.

In [55]:
Counter.value

0

In [56]:
c.__dict__

{'value': 1}

The class attribute is still `0`.

The instance now stores its own `value = 1`.

Why?

Conceptually, augmented assignment behaves like:

1. read `c.value`,
2. compute a new value,
3. assign that new value back to `c.value`.

The read can come from the class, but the assignment writes to the instance.

### Challenge

A developer wants every instance to contribute to one shared count.

Fix the implementation.

In [57]:
# Solution A: explicitly mutate the class

class SharedCounter:
    value = 0

    def increment(self):
        SharedCounter.value += 1

x = SharedCounter()
y = SharedCounter()

x.increment()
y.increment()

SharedCounter.value

2

This works when the design intentionally uses exactly `SharedCounter`.

But if subclasses should maintain their own counters, a class method is often more appropriate.

In [58]:
# Solution B: subclass-aware counting

class PerClassCounter:
    value = 0

    @classmethod
    def increment_count(cls):
        cls.value += 1

    def increment(self):
        type(self).increment_count()

In [59]:
class AlphaCounter(PerClassCounter):
    pass

class BetaCounter(PerClassCounter):
    pass

AlphaCounter().increment()
AlphaCounter().increment()
BetaCounter().increment()

PerClassCounter.value, AlphaCounter.value, BetaCounter.value

(0, 2, 1)

Notice that the subclasses now have their own `value` entries.

In [60]:
"value" in AlphaCounter.__dict__, "value" in BetaCounter.__dict__

(True, True)

This happens because `cls.value += 1` reads the inherited value and then writes a new value directly to `cls`.

# Problem 7 — The Mutable Class Attribute Trap

Class attributes are shared.

That is useful for constants and intentionally shared state.

It is dangerous when we accidentally put per-instance mutable state there.

In [61]:
class BadCart:
    items = []

Create two carts.

In [62]:
cart_a = BadCart()
cart_b = BadCart()

Append through only one instance.

In [63]:
cart_a.items.append("book")

In [64]:
cart_a.items, cart_b.items

(['book'], ['book'])

Both instances see the same list.

Let's prove that they are literally referring to the same object.

In [65]:
cart_a.items is cart_b.items

True

The list belongs to the class, not to either individual instance.

### Challenge

Rewrite the class so every cart receives an independent list.

In [66]:
# Solution

class Cart:
    def __init__(self):
        self.items = []

cart_a = Cart()
cart_b = Cart()

cart_a.items.append("book")

cart_a.items, cart_b.items

(['book'], [])

In [67]:
cart_a.items is cart_b.items

False

### A valid use of mutable class state

Sometimes sharing is exactly what we want.

For example, one registry may be shared by all instances.

In [68]:
class EventRegistry:
    event_types = set()

    @classmethod
    def register(cls, name):
        cls.event_types.add(name)

In [69]:
EventRegistry.register("created")
EventRegistry.register("deleted")
EventRegistry.event_types

{'created', 'deleted'}

### Takeaway

Ask one question before placing a mutable object on a class:

> Should every instance see mutations to the same object?

If the answer is no, create the mutable value per instance.

# Problem 8 — Inherited Class Attributes Are Accessible but Not Local

Now we extend the same namespace ideas to inheritance.

In [70]:
class BaseClient:
    timeout = 30

class ApiClient(BaseClient):
    pass

The subclass can access `timeout`.

In [71]:
ApiClient.timeout

30

But is it stored directly on `ApiClient`?

In [72]:
"timeout" in ApiClient.__dict__

False

No.

Check the base class instead.

In [73]:
"timeout" in BaseClient.__dict__

True

This gives us another example of the difference between:

- attribute resolution,
- direct namespace membership.

### Challenge

Write `find_owner(cls, name)` that returns the first class in the MRO whose direct namespace contains the attribute.

In [74]:
# Solution

def find_owner(cls, name):
    for candidate in cls.__mro__:
        if name in candidate.__dict__:
            return candidate
    return None

In [75]:
find_owner(ApiClient, "timeout").__name__

'BaseClient'

Now test it with a missing attribute.

In [76]:
find_owner(ApiClient, "missing")

# Problem 9 — Subclass Overrides and Deletion

A subclass can override an inherited class attribute simply by defining the same name.

In [77]:
class BaseSerializer:
    format = "json"

class CsvSerializer(BaseSerializer):
    format = "csv"

In [78]:
BaseSerializer.format, CsvSerializer.format

('json', 'csv')

Now inspect direct namespace membership.

In [79]:
"format" in BaseSerializer.__dict__, "format" in CsvSerializer.__dict__

(True, True)

Both classes store their own values.

Changing the base class no longer changes the subclass's resolved value.

In [80]:
BaseSerializer.format = "msgpack"
BaseSerializer.format, CsvSerializer.format

('msgpack', 'csv')

### Step 1 — Remove the override

Delete the subclass's local attribute.

In [81]:
del CsvSerializer.format

What happens now?

In [82]:
CsvSerializer.format

'msgpack'

The subclass did not become unable to resolve `format`.

Instead, deletion removed the local override, so lookup continued to the base class.

### Challenge

Create `reset_override(cls, name)` that deletes a local override if present, but leaves inherited attributes untouched.

In [83]:
# Solution

def reset_override(cls, name):
    if name in cls.__dict__:
        delattr(cls, name)
        return True
    return False

In [84]:
class Policy:
    retries = 3

class StrictPolicy(Policy):
    retries = 10

StrictPolicy.retries

10

In [85]:
reset_override(StrictPolicy, "retries")

True

In [86]:
StrictPolicy.retries

3

In [87]:
reset_override(StrictPolicy, "retries")

False

The second call returns `False` because the name is no longer direct on the subclass.

# Problem 10 — Why `hasattr` Is Not Enough for Namespace Surgery

Suppose a subclass inherits an attribute:

In [88]:
class Parent:
    enabled = True

class Child(Parent):
    pass

`hasattr` says the attribute exists:

In [89]:
hasattr(Child, "enabled")

True

But it does not live directly on `Child`:

In [90]:
"enabled" in Child.__dict__

False

What if we try to delete it from the subclass anyway?

In [91]:
try:
    delattr(Child, "enabled")
except AttributeError as exc:
    print(type(exc).__name__ + ":", exc)

AttributeError: type object 'Child' has no attribute 'enabled'


The deletion fails because `delattr(Child, "enabled")` tries to delete the attribute from `Child` itself.

The value was inherited from `Parent`.

### Challenge

Write a utility `remove_local_only(cls, *names)` that deletes only names that exist directly in `cls.__dict__`.

Return a dictionary showing which names were removed.

In [92]:
# Solution

def remove_local_only(cls, *names):
    result = {}

    for name in names:
        if name in cls.__dict__:
            delattr(cls, name)
            result[name] = True
        else:
            result[name] = False

    return result

In [93]:
class Base:
    inherited = 1

class Target(Base):
    local_a = 2
    local_b = 3

remove_local_only(Target, "inherited", "local_a", "missing")

{'inherited': False, 'local_a': True, 'missing': False}

In [94]:
Target.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 4,
              'local_b': 3,
              '__static_attributes__': (),
              '__doc__': None})

# Problem 11 — Following the Method Resolution Order

With multiple inheritance, more than one base class may define the same attribute.

In [95]:
class Left:
    source = "left"

class Right:
    source = "right"

class Combined(Left, Right):
    pass

Which value does `Combined.source` resolve?

In [96]:
Combined.source

'left'

Now inspect the MRO:

In [97]:
Combined.__mro__

(__main__.Combined, __main__.Left, __main__.Right, object)

Python searches according to the method resolution order.

The first class in that order whose namespace supplies the relevant attribute wins.

### Step 1 — Reverse the bases

Create another class with the base order reversed.

In [98]:
class ReversedCombined(Right, Left):
    pass

In [99]:
ReversedCombined.source

'right'

In [100]:
[c.__name__ for c in ReversedCombined.__mro__]

['ReversedCombined', 'Right', 'Left', 'object']

### Challenge

Write `trace_lookup(cls, name)` that returns a list showing whether each class in the MRO stores the name directly.

In [101]:
# Solution

def trace_lookup(cls, name):
    return [
        {
            "class": candidate.__name__,
            "contains_name": name in candidate.__dict__,
            "direct_value": candidate.__dict__.get(name, "<not here>"),
        }
        for candidate in cls.__mro__
    ]

In [102]:
trace_lookup(Combined, "source")

[{'class': 'Combined', 'contains_name': False, 'direct_value': '<not here>'},
 {'class': 'Left', 'contains_name': True, 'direct_value': 'left'},
 {'class': 'Right', 'contains_name': True, 'direct_value': 'right'},
 {'class': 'object', 'contains_name': False, 'direct_value': '<not here>'}]

This is an excellent debugging technique when class attributes behave unexpectedly in an inheritance hierarchy.

# Problem 12 — Methods Are Stored in the Class Namespace Too

A method definition inside a class body also creates a class attribute.

In [103]:
class Greeter:
    prefix = "Hello"

    def greet(self, name):
        return f"{self.prefix}, {name}!"

Look directly in the namespace.

In [104]:
Greeter.__dict__["greet"]

<function __main__.Greeter.greet(self, name)>

The stored value is a function.

Now access it through the class:

In [105]:
Greeter.greet

<function __main__.Greeter.greet(self, name)>

And through an instance:

In [106]:
g = Greeter()
g.greet

<bound method Greeter.greet of <__main__.Greeter object at 0x00000205966BEF90>>

The instance access produces a bound method.

That means ordinary attribute access may apply behavior that direct dictionary lookup does not.

### Challenge

Call the same underlying behavior in all three ways:

1. raw function from `__dict__`,
2. through the class,
3. through the instance.

In [107]:
# Solution

raw_function = Greeter.__dict__["greet"]

result_1 = raw_function(g, "Ada")
result_2 = Greeter.greet(g, "Grace")
result_3 = g.greet("Linus")

result_1, result_2, result_3

('Hello, Ada!', 'Hello, Grace!', 'Hello, Linus!')

All three ultimately invoke the same function logic, but the calling conventions differ because instance access creates a bound method.

# Problem 13 — Runtime Method Replacement

Because methods are class attributes, we can replace them at runtime.

In [108]:
class Formatter:
    def format(self, text):
        return text.lower()

f = Formatter()
f.format("Hello")

'hello'

Define a new function outside the class.

In [109]:
def loud_format(self, text):
    return text.upper() + "!"

Now assign it to the class.

In [110]:
Formatter.format = loud_format

Even an instance that already existed before the assignment sees the new behavior.

In [111]:
f.format("Hello")

'HELLO!'

Why?

Because the instance does not store its own `format` attribute. Each access resolves the current class attribute.

### Challenge

Attach a brand-new method under a dynamically chosen name using `setattr`.

In [112]:
# Solution

class Calculator:
    factor = 3

def multiply(self, value):
    return self.factor * value

method_name = "scale"

setattr(Calculator, method_name, multiply)

calc = Calculator()
calc.scale(10)

30

In [113]:
"scale" in Calculator.__dict__

True

### Design note

Runtime method patching is powerful, but it makes behavior harder to reason about.

For normal application architecture, explicit subclassing or dependency injection is usually easier to maintain.

# Problem 14 — Building a Class Namespace Diff

When runtime code mutates a class, it is often useful to inspect exactly what changed.

We will build that tool in steps.

### Step 1 — Capture only public direct values

Start with a function that ignores dunder entries.

In [114]:
def public_namespace_snapshot(cls):
    return {
        name: value
        for name, value in cls.__dict__.items()
        if not name.startswith("__")
    }

In [115]:
class Pipeline:
    mode = "batch"
    retries = 2

before = public_namespace_snapshot(Pipeline)
before

{'mode': 'batch', 'retries': 2}

### Step 2 — Mutate the class

In [116]:
Pipeline.mode = "stream"
Pipeline.timeout = 15
del Pipeline.retries

In [117]:
after = public_namespace_snapshot(Pipeline)
after

{'mode': 'stream', 'timeout': 15}

### Step 3 — Compare the keys

There are three interesting categories:

- added names,
- removed names,
- names present in both snapshots whose values changed.

In [118]:
before_keys = set(before)
after_keys = set(after)

added_names = after_keys - before_keys
removed_names = before_keys - after_keys
common_names = before_keys & after_keys

added_names, removed_names, common_names

({'timeout'}, {'retries'}, {'mode'})

### Challenge

Wrap the logic in `namespace_diff(before, after)`.

In [119]:
# Solution

def namespace_diff(before, after):
    before_keys = set(before)
    after_keys = set(after)

    added = {
        name: after[name]
        for name in sorted(after_keys - before_keys)
    }

    removed = {
        name: before[name]
        for name in sorted(before_keys - after_keys)
    }

    changed = {
        name: (before[name], after[name])
        for name in sorted(before_keys & after_keys)
        if before[name] != after[name]
    }

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
    }

In [120]:
namespace_diff(before, after)

{'added': {'timeout': 15},
 'removed': {'retries': 2},
 'changed': {'mode': ('batch', 'stream')}}

This pattern is useful in tests that intentionally monkey-patch classes and want to verify the exact mutation.

# Problem 15 — Reversible Class Configuration

Temporary class mutation is common in tests.

A good temporary mutation tool must restore the original state even if the code inside the temporary section fails.

We will build this incrementally.

In [121]:
class RequestConfig:
    timeout = 10
    retries = 2

Suppose we temporarily want:

In [122]:
temporary_values = {
    "timeout": 1,
    "debug": True,
}

The challenge is that `debug` does not currently exist, while `timeout` does.

So restoration needs to distinguish:

- "this attribute originally had a value",
- "this attribute originally did not exist".

### Step 1 — Record previous state

Use a sentinel object for missing values.

In [123]:
_SENTINEL = object()

previous = {
    name: getattr(RequestConfig, name, _SENTINEL)
    for name in temporary_values
}

previous

{'timeout': 10, 'debug': <object at 0x2058656d7d0>}

### Step 2 — Apply the temporary values

In [124]:
for name, value in temporary_values.items():
    setattr(RequestConfig, name, value)

RequestConfig.timeout, RequestConfig.debug

(1, True)

### Step 3 — Restore

For a previously missing value, delete the attribute.

For an existing value, assign the previous value back.

In [125]:
for name, old_value in previous.items():
    if old_value is _SENTINEL:
        delattr(RequestConfig, name)
    else:
        setattr(RequestConfig, name, old_value)

In [126]:
RequestConfig.timeout, hasattr(RequestConfig, "debug")

(10, False)

### Challenge

Convert this idea into a context manager so cleanup is guaranteed.

In [127]:
# Solution

from contextlib import contextmanager

@contextmanager
def temporary_class_attributes(cls, **changes):
    missing = object()

    previous = {
        name: getattr(cls, name, missing)
        for name in changes
    }

    try:
        for name, value in changes.items():
            setattr(cls, name, value)

        yield cls

    finally:
        for name, old_value in previous.items():
            if old_value is missing:
                delattr(cls, name)
            else:
                setattr(cls, name, old_value)

Let's test normal execution first.

In [128]:
with temporary_class_attributes(
    RequestConfig,
    timeout=99,
    debug=True,
):
    print(RequestConfig.timeout, RequestConfig.debug)

print(RequestConfig.timeout, hasattr(RequestConfig, "debug"))

99 True
10 False


Now test restoration after an exception.

In [129]:
try:
    with temporary_class_attributes(
        RequestConfig,
        timeout=123,
        tracing=True,
    ):
        print(RequestConfig.timeout, RequestConfig.tracing)
        raise RuntimeError("simulated failure")
except RuntimeError:
    pass

print(RequestConfig.timeout, hasattr(RequestConfig, "tracing"))

123 True
10 False


# Problem 16 — Per-Subclass Mutable State

We already saw that mutable class attributes are shared.

Inheritance adds another layer.

Consider this base class:

In [130]:
class Processor:
    options = {
        "verbose": False,
    }

Two subclasses inherit the same dictionary.

In [131]:
class CsvProcessor(Processor):
    pass

class JsonProcessor(Processor):
    pass

In [132]:
CsvProcessor.options is JsonProcessor.options

True

Now mutate the dictionary through one subclass.

In [133]:
CsvProcessor.options["delimiter"] = ","

In [134]:
JsonProcessor.options

{'verbose': False, 'delimiter': ','}

The mutation leaked because both subclasses resolve the same dictionary object.

### Challenge

Use `__init_subclass__` so every subclass receives its own dictionary copy.

In [135]:
# Solution

class IsolatedProcessor:
    options = {
        "verbose": False,
    }

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        cls.options = dict(cls.options)

In [136]:
class CsvIsolated(IsolatedProcessor):
    pass

class JsonIsolated(IsolatedProcessor):
    pass

In [137]:
CsvIsolated.options is JsonIsolated.options

False

In [138]:
CsvIsolated.options["delimiter"] = ","
JsonIsolated.options["indent"] = 2

In [139]:
CsvIsolated.options, JsonIsolated.options

({'verbose': False, 'delimiter': ','}, {'verbose': False, 'indent': 2})

Each subclass now stores its own `options` attribute directly.

In [140]:
"options" in CsvIsolated.__dict__, "options" in JsonIsolated.__dict__

(True, True)

# Problem 17 — Enforcing Required Class Attributes

Frameworks often require subclasses to declare class-level metadata.

For example, every plugin may need its own `kind`.

A naive test might use:

In [141]:
class PluginBase:
    kind = "generic"

In [142]:
class IncompletePlugin(PluginBase):
    pass

In [143]:
hasattr(IncompletePlugin, "kind")

True

That returns `True`, because the subclass inherits `kind`.

But suppose the requirement is stronger:

> Every concrete subclass must define its **own** `kind`.

Then `hasattr` is the wrong test.

We need direct namespace membership.

In [144]:
"kind" in IncompletePlugin.__dict__

False

### Challenge

Use `__init_subclass__` to enforce the rule automatically.

In [145]:
# Solution

class StrictPlugin:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)

        if "kind" not in cls.__dict__:
            raise TypeError(
                f"{cls.__name__} must define its own class attribute 'kind'"
            )

In [146]:
class JsonPlugin(StrictPlugin):
    kind = "json"

JsonPlugin.kind

'json'

In [147]:
try:
    class BrokenPlugin(JsonPlugin):
        pass
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)

TypeError: BrokenPlugin must define its own class attribute 'kind'


The key idea is subtle but important:

- `hasattr(cls, "kind")` asks whether lookup can resolve the name,
- `"kind" in cls.__dict__` asks whether this exact class defined/stores it directly.

# Problem 18 — Designing a Safe Class-Level Registry

A registry is one of the situations where intentionally shared mutable class state can be useful.

We will build one carefully.

### Step 1 — Start with a private registry

In [148]:
class PluginRegistry:
    _registry = {}

### Step 2 — Register by a class attribute

Every plugin class will define `kind`.

In [149]:
class JsonOutput:
    kind = "json"

class CsvOutput:
    kind = "csv"

Add a registration method.

In [150]:
class PluginRegistry:
    _registry = {}

    @classmethod
    def register(cls, plugin_cls):
        kind = getattr(plugin_cls, "kind", None)

        if not isinstance(kind, str) or not kind:
            raise ValueError(
                "Plugin must define a non-empty string class attribute 'kind'"
            )

        if kind in cls._registry:
            raise KeyError(f"Duplicate plugin kind: {kind!r}")

        cls._registry[kind] = plugin_cls

In [151]:
PluginRegistry.register(JsonOutput)
PluginRegistry.register(CsvOutput)

In [152]:
PluginRegistry._registry

{'json': __main__.JsonOutput, 'csv': __main__.CsvOutput}

### Step 3 — Avoid exposing mutable internal state

If callers receive the original dictionary, they can mutate the registry without validation.

So expose a copy instead.

In [153]:
class SafePluginRegistry:
    _registry = {}

    @classmethod
    def register(cls, plugin_cls):
        kind = getattr(plugin_cls, "kind", None)

        if not isinstance(kind, str) or not kind:
            raise ValueError(
                "Plugin must define a non-empty string class attribute 'kind'"
            )

        if kind in cls._registry:
            raise KeyError(f"Duplicate plugin kind: {kind!r}")

        cls._registry[kind] = plugin_cls

    @classmethod
    def resolve(cls, kind):
        try:
            return cls._registry[kind]
        except KeyError:
            raise KeyError(f"Unknown plugin kind: {kind!r}") from None

    @classmethod
    def snapshot(cls):
        return dict(cls._registry)

In [154]:
SafePluginRegistry.register(JsonOutput)
SafePluginRegistry.register(CsvOutput)

snapshot = SafePluginRegistry.snapshot()
snapshot

{'json': __main__.JsonOutput, 'csv': __main__.CsvOutput}

Mutating the snapshot does not mutate the real registry.

In [155]:
snapshot["fake"] = object

"fake" in SafePluginRegistry.snapshot()

False

### Challenge

Add an `unregister` method and test duplicate registration.

In [156]:
# Solution extension

def _unregister(cls, kind):
    try:
        return cls._registry.pop(kind)
    except KeyError:
        raise KeyError(f"Unknown plugin kind: {kind!r}") from None

SafePluginRegistry.unregister = classmethod(_unregister)

In [157]:
removed = SafePluginRegistry.unregister("csv")
removed.__name__

'CsvOutput'

In [158]:
SafePluginRegistry.snapshot()

{'json': __main__.JsonOutput}

In [159]:
try:
    SafePluginRegistry.register(JsonOutput)
except KeyError as exc:
    print(type(exc).__name__ + ":", exc)

KeyError: "Duplicate plugin kind: 'json'"


# Problem 19 — Auditing Where an Attribute Comes From

At this point we have several possible origins for a name:

- directly on an instance,
- directly on its class,
- inherited from a base class,
- or accessible through broader class-object behavior.

Let's build a diagnostic utility.

In [160]:
class AuditBase:
    inherited = "base"

class AuditChild(AuditBase):
    local = "child"

audit_instance = AuditChild()
audit_instance.instance_only = "instance"

### Step 1 — Find instance-local storage

In [161]:
"instance_only" in audit_instance.__dict__

True

### Step 2 — Find class-local storage

In [162]:
"local" in AuditChild.__dict__

True

### Step 3 — Find the defining class for inherited values

In [163]:
find_owner(AuditChild, "inherited").__name__

'AuditBase'

### Challenge

Write `explain_attribute(obj, name)`.

For an ordinary instance, report:

- resolved value,
- whether the name exists directly on the instance,
- the first class in the instance type's MRO whose direct namespace contains it.

In [164]:
# Solution

def explain_attribute(obj, name):
    instance_dict = getattr(obj, "__dict__", {})
    cls = type(obj)
    owner = find_owner(cls, name)

    return {
        "name": name,
        "resolved": getattr(obj, name, "<missing>"),
        "instance_local": name in instance_dict,
        "class_owner": None if owner is None else owner.__name__,
    }

In [165]:
explain_attribute(audit_instance, "instance_only")

{'name': 'instance_only',
 'resolved': 'instance',
 'instance_local': True,
 'class_owner': None}

In [166]:
explain_attribute(audit_instance, "local")

{'name': 'local',
 'resolved': 'child',
 'instance_local': False,
 'class_owner': 'AuditChild'}

In [167]:
explain_attribute(audit_instance, "inherited")

{'name': 'inherited',
 'resolved': 'base',
 'instance_local': False,
 'class_owner': 'AuditBase'}

In [168]:
explain_attribute(audit_instance, "missing")

{'name': 'missing',
 'resolved': '<missing>',
 'instance_local': False,
 'class_owner': None}

# Problem 20 — Capstone: A Reversible Subclass Configuration System

We will combine several ideas from the notebook.

We want a base configuration class with defaults:

In [169]:
class ServiceDefaults:
    timeout = 30
    retries = 3
    tracing = False

A subclass should be able to override selected values:

In [170]:
class PaymentService(ServiceDefaults):
    timeout = 10

In [171]:
PaymentService.timeout, PaymentService.retries, PaymentService.tracing

(10, 3, False)

Notice:

- `timeout` is local to `PaymentService`,
- `retries` and `tracing` are inherited.

In [172]:
{
    name: name in PaymentService.__dict__
    for name in ("timeout", "retries", "tracing")
}

{'timeout': True, 'retries': False, 'tracing': False}

We now want a small configuration API with four operations:

1. `read(name)` — resolve the current value,
2. `override(name, value)` — create/update a subclass-local override,
3. `reset(name)` — remove only a local override so inheritance is visible again,
4. `origin(name)` — report which class in the MRO currently defines the value.

Unknown configuration names should be rejected.

### Step 1 — Decide which names are allowed

For this exercise, the allowed configuration names are the public non-callable attributes inherited through the class hierarchy.

We will use a helper to search the MRO.

In [173]:
def allowed_config_names(cls):
    names = set()

    for candidate in cls.__mro__:
        for name, value in candidate.__dict__.items():
            if name.startswith("_"):
                continue
            if callable(value):
                continue
            names.add(name)

    return names

In [174]:
allowed_config_names(PaymentService)

{'retries', 'timeout', 'tracing'}

### Step 2 — Build the API

In [175]:
# Solution

class ConfigurableService(ServiceDefaults):
    @classmethod
    def _validate_name(cls, name):
        if name not in allowed_config_names(cls):
            raise KeyError(f"Unknown configuration name: {name!r}")

    @classmethod
    def read(cls, name):
        cls._validate_name(name)
        return getattr(cls, name)

    @classmethod
    def override(cls, name, value):
        cls._validate_name(name)
        setattr(cls, name, value)

    @classmethod
    def reset(cls, name):
        cls._validate_name(name)

        if name in cls.__dict__:
            delattr(cls, name)

    @classmethod
    def origin(cls, name):
        cls._validate_name(name)
        owner = find_owner(cls, name)
        return None if owner is None else owner.__name__

Create a concrete service.

In [176]:
class SearchService(ConfigurableService):
    timeout = 5

Read current values.

In [177]:
SearchService.read("timeout"), SearchService.read("retries")

(5, 3)

Inspect their origins.

In [178]:
SearchService.origin("timeout"), SearchService.origin("retries")

('SearchService', 'ServiceDefaults')

Override an inherited setting.

In [179]:
SearchService.override("retries", 9)

In [180]:
SearchService.retries, SearchService.origin("retries")

(9, 'SearchService')

The override is now stored directly on `SearchService`.

In [181]:
"retries" in SearchService.__dict__

True

Reset the override.

In [182]:
SearchService.reset("retries")

In [183]:
SearchService.retries, SearchService.origin("retries")

(3, 'ServiceDefaults')

The inherited value is visible again.

Finally, verify that unknown names are rejected.

In [184]:
try:
    SearchService.read("does_not_exist")
except KeyError as exc:
    print(type(exc).__name__ + ":", exc)

KeyError: "Unknown configuration name: 'does_not_exist'"


### Capstone conclusion

This final design used nearly every major idea from the notebook:

- direct namespace membership,
- normal attribute resolution,
- dynamic mutation,
- deletion,
- inheritance,
- subclass shadowing,
- MRO inspection,
- and careful validation.

The most important reasoning pattern is still the same:

> Before changing an attribute, determine **where it currently lives** and **where the assignment will write**.

# Extra Tutorial Drills

These are shorter exercises, but still follow the same "predict → run → explain" workflow.

Try to predict every output before executing the next cell.

## Drill 1 — Dotted access and `getattr`

Both should resolve the same ordinary class attribute.

In [185]:
class Drill:
    value = 100

Drill.value, getattr(Drill, "value")

(100, 100)

Now use the three-argument form of `getattr` for a missing attribute.

In [186]:
getattr(Drill, "missing", "fallback")

'fallback'

## Drill 2 — Dynamic creation

In [187]:
name = "dynamic"
setattr(Drill, name, 200)

Drill.dynamic

200

In [188]:
Drill.__dict__["dynamic"]

200

## Drill 3 — Dynamic deletion

In [189]:
delattr(Drill, "dynamic")

hasattr(Drill, "dynamic")

False

## Drill 4 — Direct namespace versus inherited lookup

In [190]:
class DrillBase:
    x = 10

class DrillChild(DrillBase):
    pass

DrillChild.x, "x" in DrillChild.__dict__, "x" in DrillBase.__dict__

(10, False, True)

## Drill 5 — Subclass shadow

In [191]:
DrillChild.x = 50

DrillBase.x, DrillChild.x

(10, 50)

In [192]:
DrillChild.__dict__["x"]

50

## Drill 6 — Remove the shadow

In [193]:
del DrillChild.x

DrillChild.x

10

## Drill 7 — Instance shadow

In [194]:
obj = DrillChild()
obj.x = 999

DrillChild.x, obj.x

(10, 999)

In [195]:
obj.__dict__

{'x': 999}

## Drill 8 — Remove the instance shadow

In [196]:
del obj.x

obj.x

10

## Drill 9 — Saved `mappingproxy` is live

In [197]:
class ProxyDemo:
    a = 1

proxy = ProxyDemo.__dict__

In [198]:
ProxyDemo.b = 2

proxy["b"]

2

## Drill 10 — Class methods and subclass-local assignment

In [199]:
class DrillCounter:
    count = 0

    @classmethod
    def bump(cls):
        cls.count += 1

class DrillSubCounter(DrillCounter):
    pass

DrillSubCounter.bump()

DrillCounter.count, DrillSubCounter.count

(0, 1)

In [200]:
"count" in DrillSubCounter.__dict__

True

# Final Review

You should now be comfortable answering questions like:

### Storage

- Is this name stored directly on the class?
- Is it inherited?
- Is it shadowed by an instance?
- Is the value a shared mutable object?

### Access

- What will dotted access resolve?
- Would `getattr` behave differently from direct `__dict__` lookup?
- Which class in the MRO supplies the value?

### Mutation

- Will assignment change the class or create a new shadow?
- Will `+=` write to the same namespace from which it read?
- Is `setattr` appropriate because the name is dynamic?

### Deletion

- Is the name actually deletable from this class?
- Will deleting it make the attribute disappear completely?
- Or will deletion reveal an inherited value?

### Design

- Should this state be shared by all instances?
- Should each subclass receive its own mutable copy?
- Is runtime class mutation clearer than subclassing?
- Should a dynamic configuration API validate names before mutation?

If you can answer those questions before running the code, you have developed a strong working model of Python class attributes.

# Suggested Study Method

For each problem in this notebook:

1. stop before the solution,
2. predict the output,
3. write down which namespace you think contains the name,
4. run the experiment,
5. inspect `__dict__`,
6. compare the result with your prediction,
7. explain whether the operation performed lookup, mutation, shadowing, or deletion.

That workflow is especially effective for Python object-model topics because many surprises come from confusing **resolution** with **storage**.